# Python-generated layered recipe

This notebook generates detached planar fiber collections in Python, inserts them one ply at a time, and places a relaxation gate after every insertion. It demonstrates how a manufacturing approximation becomes an ordered TANGLE recipe.

In [ ]:
import math
import random
from pathlib import Path
import tangle

output = Path("output")
output.mkdir(exist_ok=True)

## Define a reusable Python generator

The generator returns an ordinary detached `FiberCollection`. It may use Python, NumPy, imported measurements, or another geometry package; TANGLE only needs the resulting centerlines and material metadata.

In [ ]:
def planar_layer(
    name: str, layer: int, count: int, seed: int
) -> tangle.FiberCollection:
    rng = random.Random(seed)
    material = tangle.Material("felt fiber", diameter=19.0e-6)
    collection = tangle.FiberCollection(name)
    for _ in range(count):
        angle = rng.uniform(0.0, math.tau)
        direction = [math.cos(angle), math.sin(angle), 0.0]
        center = [
            rng.uniform(0.2e-3, 0.8e-3),
            rng.uniform(0.2e-3, 0.8e-3),
            0.0,
        ]
        half_length = 0.35e-3
        collection.add_fiber(
            [
                [center[i] - half_length * direction[i] for i in range(3)],
                [center[i] + half_length * direction[i] for i in range(3)],
            ],
            material,
            formation_layer=layer,
            tags={"ply": str(layer)},
        )
    return collection

## Compose the formation recipe

Each `insert()` assigns a formation step. The full topology is packed before execution, but later plies remain dormant until their activation operation. This permits insert → relax → insert → relax ordering without rebuilding the GPU world.

In [ ]:
# The plies stack along z, the cell's default stack axis.
recipe = tangle.Recipe(tangle.Cell([1.0e-3, 1.0e-3, 2.0e-3]))
for layer in range(3):
    recipe.insert(
        planar_layer(f"ply_{layer}", layer, count=12, seed=100 + layer),
        translation=[0.0, 0.0, (layer + 1) * 0.4e-3],
    )
    recipe.relax_until_converged(max_iterations=2_000)

for operation in recipe.operations():
    print(operation)

## Configure and execute one GPU-resident run

The `fast` adaptive profile is only a starting point. Pass keyword changes to `profile()`, or call `replace()` on its result, to adjust the refinement and coarsening fields.

In [ ]:
settings = tangle.RelaxationSettings(
    max_iterations=10_000,
    penetration_tolerance=0.1e-6,
    max_step=2.0e-6,
    adaptive_segmentation=tangle.AdaptiveSegmentationSettings.profile("fast"),
)
settings.to_dict()

In [ ]:
result = recipe.run(settings)
print(result)
print(f"final fiber count: {result.fiber_count}")

In [ ]:
result.write_ovito(
    output / "layered_recipe.dump",
    view_script_path=output / "layered_recipe_view.py",
    session_path=output / "layered_recipe.ovito",
)